In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src.utils.config import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

# 01 Data cleaning
Every cleaning decision is made in `src/data_processing/clean.py` and logged. This notebook shows the **problem -> fix** evidence for each item in the PRD checklist (section 9.1).

In [2]:
from src.data_processing import clean
log = {}
raw = pd.read_csv(RAW/'customers.csv'); cl = pd.read_csv(PROCESSED/'customers.csv')
print('customers raw:', len(raw), '-> clean:', len(cl), '| exact duplicates removed:', len(raw)-len(raw.drop_duplicates()))
print('country spellings raw:', raw.country.nunique(), '-> clean:', cl.country.nunique())
print(raw.country.value_counts().head(8).to_dict())

customers raw: 8048 -> clean: 8000 | exact duplicates removed: 48
country spellings raw: 83 -> clean: 20
{'United States': 1262, 'India': 1071, 'United Kingdom': 530, 'Canada': 365, 'Brazil': 336, 'Germany': 324, 'France': 267, 'Mexico': 254}


## Country: 83 spellings -> 20 canonical countries
`INDIA / india / IN / Bharat` all become `India`. 409 missing countries were **filled from each customer's most common login country** (the two agree 92% of the time where both exist), not guessed.

In [3]:
print(json.dumps(json.load(open(REPORTS/'cleaning_log.json')), indent=1))

{
 "customers_raw_rows": 8048,
 "customers_exact_duplicates_removed": 48,
 "country_missing_before": 409,
 "country_filled_from_login_location": 409,
 "age_outliers_set_to_null": 21,
 "age_imputed_with_segment_median": 342,
 "language_missing_filled_by_country_mode": 251,
 "subscriptions_price_differs_from_plan_list_price": 570,
 "device_missing": 664,
 "watch_duration_minutes_outliers_capped": 161,
 "session_duration_minutes_outliers_capped": 161,
 "sessions_shorter_than_watch_time_fixed": 6068,
 "completion_missing": 531,
 "payments_linked_to_subscription": 110735,
 "payments_unlinked": 0,
 "payment_amount_anomalies_replaced_with_sub_price": 1112,
 "tickets_invalid_dates_set_to_null": 61,
 "feedback_sentiment_labels_disagreeing_with_rating": 263,
 "rows_customers": 8000,
 "rows_subscription_plans": 3,
 "rows_subscriptions": 8875,
 "rows_content": 500,
 "rows_viewing_activity": 66422,
 "rows_payments": 110735,
 "rows_support_tickets": 6120,
 "rows_customer_feedback": 5046,
 "rows_chur

## Outliers and missing values: strategy per column
| Column | Problem | Strategy | Why |
|---|---|---|---|
| age | 322 missing, 21 impossible (2, 5, 130, 145) | outliers -> null, then impute **segment median**; flag columns kept | students are younger than corporate users, so a global median would bias |
| country | 409 missing | modal login country | behavioural evidence beats a constant |
| city | 484 missing | 'Unknown' | no reliable source |
| preferred_language | 254 missing | modal language of the customer's country | |
| device_type | 664 missing + 24 spellings | canonical set of 5; missing -> customer's modal device | |
| completion_percentage | 531 missing | median by content type | |
| watch/session minutes | 161 values of 500-900 min | capped at the 99.5th pct of normal values; flag kept | genuine binges do not last 15 hours |
| session < watch time | 6,068 rows (9%) | session raised to watch time | a session cannot be shorter than what was watched |
| payments.amount | 1,112 anomalies (0, x2, x0.5, x11...) | replaced by the subscription's price; raw kept in `amount_raw` | 99% of payments equal the subscription price |
| ticket_date | 61 invalid ('2025-13-45') | set to NULL, never guessed | excluded from time analyses |
| CSAT / resolution time blank | 1,528 open tickets | **left blank on purpose** | blank = not yet resolved, not missing data |
| sentiment_label | ~5% contradict the star rating | `sentiment_validated` derived from rating (>=4 pos, 3 neu, <=2 neg) | original kept |

In [4]:
v = pd.read_csv(RAW/'viewing_activity.csv'); vc = pd.read_csv(PROCESSED/'viewing_activity.csv')
print('device values raw:', v.device_type.nunique(), '-> clean:', vc.device_type.nunique())
print('max watch minutes raw:', v.watch_duration_minutes.max(), '-> clean:', vc.watch_duration_minutes.max())
p = pd.read_csv(PROCESSED/'payments.csv')
print('payments linked to a subscription:', p.subscription_id.notna().mean())
print(p[p.is_amount_anomaly][['payment_id','amount_raw','amount']].head())

device values raw: 23 -> clean: 6
max watch minutes raw: 894 -> clean: 144
payments linked to a subscription: 1.0
      payment_id  amount_raw  amount
5    PAY00000006       11.49   22.99
71   PAY00000072      252.89   22.99
95   PAY00000096      170.39   15.49
111  PAY00000112        0.00   15.49
640  PAY00000641      170.39   15.49


## Finding that changes how the data must be read
- `subscriptions.monthly_price` **never differs between a customer's two rows**, so it is not the price at the time of each segment (the dictionary says it is). Real price changes only exist as plan changes (see SQL Q29).
- Churn dates (Oct-25 to Jun-26) sit **inside** the viewing window, and 36% of churners still have sessions after their churn date. Features therefore cannot be made perfectly 'before churn'. See the final report, section Limitations.